<a href="https://colab.research.google.com/github/22111950-creator/YOLO-Study/blob/COLAB_TEST/test_trash_%ED%95%99%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Sat May 30 12:28:52 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
import os
HOME = os.getcwd()
print(HOME)

/content


In [4]:
# Pip install method (recommended)

!pip install ultralytics==8.2.103 -q

from IPython import display
display.clear_output()

# prevent ultralytics from tracking your activity
!yolo settings sync=False

import ultralytics
ultralytics.checks()

Ultralytics YOLOv8.2.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 47.5/112.6 GB disk)


In [5]:
from ultralytics import YOLO

from IPython.display import display, Image

In [2]:
!mkdir -p {HOME}/datasets
%cd {HOME}/datasets

!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="YxAQrthA56eou3zmff0I")
project = rf.workspace("s-workspace-f8twe").project("test-trash-t0ugp-eyhto")
version = project.version(1)
dataset = version.download("yolov8")


/content/{HOME}/datasets/{HOME}/datasets
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Test-trash-1 in yolov8:: 100%|██████████| 12153/12153 [00:02<00:00, 4256.81it/s]


In [ ]:
# 1. HOME 변수로 이동 (파이썬 변수 사용 시 $ 붙임)
%cd $HOME

# 2. YOLO 학습 진행 ($dataset.location 사용)
!yolo task=detect mode=train model=yolov8s.pt data=$dataset.location/data.yaml epochs=100 imgsz=640 plots=True

[Errno 2] No such file or directory: '$HOME'
/content/{HOME}/datasets/{HOME}/datasets
100% 21.5M/21.5M [00:00<00:00, 174MB/s]
New https://pypi.org/project/ultralytics/8.4.57 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.103 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: task=detect, mode=train, model=yolov8s.pt, data=/content/{HOME}/datasets/{HOME}/datasets/Test-trash-1/data.yaml, epochs=100, time=None, patience=100, batch=16, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=None, name=train, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True,

In [ ]:
# ONNX 변환 및 검증에 필요한 최신 라이브러리 설치
!pip uninstall ultralytics -y
!pip install ultralytics==8.0.196

Found existing installation: ultralytics 8.0.196
Uninstalling ultralytics-8.0.196:
  Successfully uninstalled ultralytics-8.0.196
  Using cached ultralytics-8.0.196-py3-none-any.whl.metadata (31 kB)
Using cached ultralytics-8.0.196-py3-none-any.whl (631 kB)


In [ ]:
import ultralytics
print(ultralytics.__version__)

8.0.196


In [ ]:
# 1. 에러가 발생한 필수 모듈 및 연산자 변환 지원 패키지 설치
!pip install onnxscript

# 설치가 완료되면 기존 변환 코드를 다시 실행합니다.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 58.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 14.2 MB/s eta 0:00:00


In [ ]:
!pip install onnxslim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 238.4/238.4 kB 5.3 MB/s eta 0:00:00


In [ ]:
import torch
import functools

# 1. PyTorch 보안 우회 패치
torch.load = functools.partial(torch.load, weights_only=False)

from ultralytics import YOLO

# 2. 모델 로드
model = YOLO("/content/{HOME}/datasets/runs/detect/train2/weights/best.pt")

# 3. opset 제한을 우회하기 위해 12 버전으로 지정하여 추출
model.export(
    format="onnx",
    imgsz=640,
    opset=12,          # [수정] 11에서 발생한 Resize 어댑터 에러를 피하기 위해 12로 변경
    simplify=True,     # 그래프 단순화 유지
    dynamic=False,
    nms=False
)

Ultralytics YOLOv8.0.196 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon 2.00GHz)
Model summary (fused): 168 layers, 11127519 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/content/{HOME}/datasets/runs/detect/train2/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 9, 8400) (21.5 MB)

ONNX: starting export with onnx 1.21.0 opset 12...
W0529 14:53:30.544000 52917 torch/onnx/_internal/exporter/_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 12 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features
/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `

'/content/{HOME}/datasets/runs/detect/train2/weights/best.onnx'

In [ ]:
import onnx

# 1. 방금 생성된 ONNX 파일 로드
onnx_model = onnx.load("/content/{HOME}/datasets/runs/detect/train2/weights/best.onnx")

# 2. 메타데이터의 opset 버전을 강제로 12로 패치
for opset in onnx_model.opset_import:
    if opset.domain == '' or opset.domain == 'ai.onnx':
        print(f"기존 버전: {opset.version} -> 변경 버전: 12")
        opset.version = 12

# 3. 변경된 내용을 같은 경로에 덮어쓰기 저장
onnx.save(onnx_model, "/content/{HOME}/datasets/runs/detect/train2/weights/best.onnx")
print("ONNX Opset 버전 강제 변경 완료! 이제 다운로드 하세요. ✅")

기존 버전: 18 -> 변경 버전: 12
ONNX Opset 버전 강제 변경 완료! 이제 다운로드 하세요. ✅


In [ ]:
import os
import random
import shutil
from google.colab import files

# [수정된 부분] /content 내부에서 train/images 폴더를 자동으로 검색합니다.
base_path = '/content'
source_dir = None

for root, dirs, files_list in os.walk(base_path):
    if root.endswith('train/images'):
        source_dir = root
        break

if source_dir is None:
    raise FileNotFoundError("Colab 환경 내에서 'train/images' 폴더를 찾을 수 없습니다. 데이터셋 다운로드가 완료되었는지 확인해주세요.")

print(f"찾은 이미지 경로: {source_dir}")

# 2. 랜덤 추출한 이미지를 저장할 새 폴더 생성
target_dir = '/content/calibration_samples'
os.makedirs(target_dir, exist_ok=True)

# 3. 원본 폴더에서 이미지 파일 목록 가져오기
all_images = [f for f in os.listdir(source_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]

# 4. 설정한 수량만큼 랜덤 추출
sample_count = min(100, len(all_images))
selected_images = random.sample(all_images, sample_count)

print(f"총 {len(all_images)}장의 이미지 중 {sample_count}장을 랜덤 추출합니다.")

# 5. 선택된 이미지를 새 폴더로 복사
for img_name in selected_images:
    source_path = os.path.join(source_dir, img_name)
    target_path = os.path.join(target_dir, img_name)
    shutil.copy(source_path, target_path)

print(f"새 폴더({target_dir})로 복사 완료.")

# 6. 복사된 폴더를 zip 파일로 압축
zip_file_path = '/content/calibration_samples.zip'
shutil.make_archive('/content/calibration_samples', 'zip', target_dir)
print(f"압축 완료: {zip_file_path}")

# 7. 내 PC로 다운로드 실행
print("로컬 다운로드를 시작합니다...")
files.download(zip_file_path)

찾은 이미지 경로: /content/{HOME}/datasets/Test-trash-1/train/images
총 4859장의 이미지 중 100장을 랜덤 추출합니다.
새 폴더(/content/calibration_samples)로 복사 완료.
압축 완료: /content/calibration_samples.zip
로컬 다운로드를 시작합니다...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

# -------------------------------------------------------------
# 업로드하신 5.3.0 버전 파일명으로 정확히 매칭해두었습니다.
# -------------------------------------------------------------
WHL_FILE_NAME = "hailo_dataflow_compiler-5.3.0-py3-none-linux_x86_64.whl"
ONNX_PATH = "/content/best.onnx"
ALLS_PATH = "/content/yolov8s.alls"
CALIB_DIR = "/content/calibration_samples"  # 100장 이미지 추출했던 폴더 경로
MODEL_NAME = "best_yolov8"

# 1. Hailo Dataflow Compiler (DFC) 패키지 설치
print("📦 Hailo DFC 패키지 설치를 시작합니다...")
if os.path.exists(f"/content/{WHL_FILE_NAME}"):
    import subprocess
    # Use subprocess to capture output and check return code for pip install
    result = subprocess.run(["pip", "install", f"/content/{WHL_FILE_NAME}"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ DFC 패키지 설치 완료.")
    else:
        print(f"❌ DFC 패키지 설치 실패. 오류: {result.stderr}")
        print("💡 제공된 wheel 파일이 유효하지 않거나 호환되지 않는 것 같습니다.")
        print("   Colab 환경과 호환되는 유효한 'hailo_dataflow_compiler' wheel 파일을 업로드했는지 확인해주세요.")
        raise ModuleNotFoundError("hailo_sdk_client 모듈을 임포트할 수 없습니다. DFC 패키지 설치에 실패했습니다.")
else:
    raise FileNotFoundError(f"❌ /content/{WHL_FILE_NAME} 파일이 아직 Colab에 업로드되지 않았습니다. 업로드 완료 후 다시 실행해주세요.")
print("-" * 50)

# 2. 패키지 로드 및 컴파일러 초기화
from hailo_sdk_client import ClientRunner
import numpy as np
from PIL import Image

runner = ClientRunner(hw_arch="hailo8")

# 3. ONNX 모델 파싱 (HAR 파일 생성)
print("🏗️ ONNX 모델 파싱 중...")
# 내부적으로 같은 폴더에 있는 best.onnx.data 가중치 파일을 자동으로 연동합니다.
runner.translate_onnx_model(
    model_path=ONNX_PATH,
    model_name=MODEL_NAME,
    end_node_names=["output0"]  # YOLOv8 기본 출력 노드명
)
print("✅ 모델 파싱 완료.")

# 4. ALLS 최적화 스크립트 적용
if os.path.exists(ALLS_PATH):
    print("📝 탐색기에서 찾은 최적화 스크립트(yolov8s.alls)를 적용합니다...")
    runner.load_model_script(ALLS_PATH)

# 5. 캘리브레이션 이미지 로드 및 전처리 (INT8 양자화용)
print("🖼️ 캘리브레이션용 이미지 데이터셋 로드 중...")
def load_calibration_data(image_dir, target_size=(640, 640)):
    images = []
    for file in os.listdir(image_dir):
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            img = Image.open(os.path.join(image_dir, file)).convert('RGB')
            img = img.resize(target_size)
            img_array = np.array(img).astype(np.float32) / 255.0
            images.append(img_array)
    return np.array(images)

calib_dataset = load_calibration_data(CALIB_DIR)
print(f"✅ 총 {len(calib_dataset)}장의 이미지가 양자화 데이터셋으로 준비되었습니다.")

# 6. 모델 최적화 및 양자화 실행
print("⚡ 모델 양자화(Optimization) 진행 중... (시간이 다소 소요될 수 있습니다)")
runner.optimize(calib_dataset)
print("✅ 양자화 완료.")

# 7. 최종 HEF 파일 컴파일 및 저장
print("🚀 HEF 파일 컴파일 중...")
hef = runner.compile()

output_hef_path = f"/content/{MODEL_NAME}.hef"
with open(output_hef_path, "wb") as f:
    f.write(hef)

print(f"🎉 모든 변환이 성공적으로 완료되었습니다! 파일 경로: {output_hef_path}")

# 8. 내 PC로 최종 결과물 자동 다운로드
from google.colab import files
files.download(output_hef_path)

📦 Hailo DFC 패키지 설치를 시작합니다...
❌ DFC 패키지 설치 실패. 오류:   error: subprocess-exited-with-error
  
  × Building wheel for pygraphviz (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pygraphviz
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pygraphviz)

💡 제공된 wheel 파일이 유효하지 않거나 호환되지 않는 것 같습니다.
   Colab 환경과 호환되는 유효한 'hailo_dataflow_compiler' wheel 파일을 업로드했는지 확인해주세요.


ModuleNotFoundError: hailo_sdk_client 모듈을 임포트할 수 없습니다. DFC 패키지 설치에 실패했습니다.